In [1]:
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,380 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,767 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

In [2]:
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from concurrent.futures import ThreadPoolExecutor

In [3]:
chrome_options = Options()
chrome_options.add_argument("--headless")  # Chạy không hiển thị trình duyệt
chrome_options.add_argument("--no-sandbox")  # Bỏ qua sandbox (bắt buộc trên Colab)
chrome_options.add_argument("--disable-dev-shm-usage")  # Giảm sử dụng bộ nhớ
chrome_options.add_argument("--disable-blink-features=AutomationControlled")  # Tránh bị phát hiện là bot
chrome_options.add_argument("--window-size=1920,1080")

In [4]:
calorie_ranges = [
    # list(range(1000, 1500, 100)),
    # list(range(1500, 2000, 100)),
    # list(range(2000, 2500, 100)),
    # list(range(2500, 3000, 100)),
    list(range(3000, 3600, 100))
]

In [5]:
def crawl_meals(calorie):
    """Hàm crawl 100 meal cho một mức calo với xử lý timeout"""
    for _ in range(3):  # Thử lại tối đa 3 lần nếu lỗi
        try:
            driver = webdriver.Chrome(options=chrome_options)
            driver.set_page_load_timeout(60)
            url = "https://www.eatthismuch.com/"
            driver.get(url)
            time.sleep(random.uniform(2, 4))

            meal_list = []
            for i in range(1000):
                print(f"🔥 Crawl {calorie} calo - Meal {i+1}/1000")

                calorie_input = driver.find_element(By.CSS_SELECTOR, "input[type='number']")
                calorie_input.clear()
                calorie_input.send_keys(str(calorie))

                generate_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
                driver.execute_script("arguments[0].scrollIntoView();", generate_button)
                time.sleep(random.uniform(1.5, 3))
                generate_button.click()

                time.sleep(random.uniform(3, 6))

                meals = driver.find_elements(By.CSS_SELECTOR, "section.svelte-bdns86 h4")
                meal_details = driver.find_elements(By.CSS_SELECTOR, "ul.svelte-1cjrqa0 > li")

                meal_data = {}

                for meal in meals:
                    meal_name = meal.text
                    meal_data[meal_name] = []

                for item in meal_details:
                    meal_name = item.find_element(By.XPATH, "./ancestor::section[contains(@class, 'svelte-bdns86')]/header/hgroup/h4").text
                    food_name = item.find_element(By.CSS_SELECTOR, "a.svelte-k5vk40").text
                    serving = item.find_element(By.CSS_SELECTOR, "p.svelte-u63x5w").text

                    meal_data[meal_name].append({
                        "food": food_name,
                        "serving": serving
                    })

                meal_list.append(meal_data)

                if i % 10 == 0:
                    time.sleep(random.uniform(5, 10))

            driver.quit()
            return calorie, meal_list

        except Exception as e:
            print(f"⚠️ Lỗi {e}, thử lại sau 10 giây...")
            time.sleep(10)

    print(f"❌ Không thể crawl {calorie} calo sau 3 lần thử")
    return calorie, "Failed"


In [6]:
# Chạy crawl từng khoảng calo và lưu thành file riêng
for calo_range in calorie_ranges:
    range_name = f"{calo_range[0]}-{calo_range[-1]}"  # Ví dụ: 1000-1400
    data = {}

    print(f"\n🚀 Bắt đầu crawl khoảng {range_name} calo...\n")

    with ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(crawl_meals, calo_range))

    # Gộp kết quả
    for calorie, meals in results:
        data[str(calorie)] = meals

    # Lưu vào file JSON
    filename = f"meal_data_{range_name}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"✅ Dữ liệu khoảng {range_name} calo đã được lưu vào '{filename}'")

print("\n🎉 Hoàn thành crawl tất cả dữ liệu!")

Streaming output truncated to the last 5000 lines.
🔥 Crawl 3200 calo - Meal 686/1000
🔥 Crawl 3300 calo - Meal 719/1000
🔥 Crawl 3100 calo - Meal 720/1000
🔥 Crawl 3000 calo - Meal 721/1000
🔥 Crawl 3200 calo - Meal 687/1000
🔥 Crawl 3400 calo - Meal 690/1000
🔥 Crawl 3300 calo - Meal 720/1000
🔥 Crawl 3100 calo - Meal 721/1000
🔥 Crawl 3200 calo - Meal 688/1000
🔥 Crawl 3400 calo - Meal 691/1000
🔥 Crawl 3300 calo - Meal 721/1000
🔥 Crawl 3200 calo - Meal 689/1000
🔥 Crawl 3000 calo - Meal 722/1000
🔥 Crawl 3100 calo - Meal 722/1000
🔥 Crawl 3200 calo - Meal 690/1000
🔥 Crawl 3300 calo - Meal 722/1000
🔥 Crawl 3400 calo - Meal 692/1000
🔥 Crawl 3000 calo - Meal 723/1000
🔥 Crawl 3100 calo - Meal 723/1000
🔥 Crawl 3300 calo - Meal 723/1000
🔥 Crawl 3200 calo - Meal 691/1000
🔥 Crawl 3400 calo - Meal 693/1000
🔥 Crawl 3000 calo - Meal 724/1000
🔥 Crawl 3100 calo - Meal 724/1000
🔥 Crawl 3300 calo - Meal 724/1000
🔥 Crawl 3400 calo - Meal 694/1000
🔥 Crawl 3000 calo - Meal 725/1000
🔥 Crawl 3200 calo - Meal 692/10